# Day 7 — Defend the complete decoder

You can name every layer. Can you explain the entire computation, detect a plausible bug, and defend what the measurements actually prove?

This is the core Day 7 synthesis session. Study it **before** the existing optional [recurrent-depth notebook](01_recurrent_depth.ipynb); the older filename is preserved for stable links. Prerequisite: Chapter 5 Days 5–6 and the baseline/modern notebooks.

Work through each deep question before revealing its runnable solution. All attempt cells are safe placeholders. Reference checks use CPU float64, seed 505, atol 1e-10 and rtol 1e-8. The final comparison contract is a proposal, not permission to run a training campaign. Passing the notebook does not automatically establish your architecture defense or language capability.

In [ ]:
from pathlib import Path
import sys, copy
import torch
from torch import nn
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/dongxi_llms/decoder_lab.py").exists())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, teaching_batch, next_token_loss,
    parameter_count, cost_estimate,
)
from dongxi_llms.decoder_audit import (
    trace_decoder, parameter_ledger, boundary_report, WrongAxisCenter,
)
torch.set_num_threads(1)
torch.manual_seed(505)
cfg = DecoderConfig(modern=True, qk_norm=True, kv_heads=2)
model = TinyDecoder(cfg).double().eval()
ids, labels = teaching_batch()
print("CPU reference:", torch.__version__, ids.shape, labels.shape)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from dongxi_llms import decoder_architecture as architecture, decoder_visuals as viz
def show_visual(figure):
    display(figure)
    plt.close(figure)

### Locate the complete model

The modern model uses token lookup, pre-norm blocks, final RMSNorm, and a tied vocabulary head. Shapes are structural labels, not measurements of speed. B=batch, T=positions, D=stream width, V=vocabulary size.

![Locate the complete model](../figures/chapter-05/day-07-02_architecture_defense-architecture-map.png)

*Saved reference preview. The next cell regenerates the figure; it does not overwrite this preview.*

In [ ]:
show_visual(architecture.model_map(focus='assembly', modern=True))

### Explain both residual updates

The block reads normalized states, then adds attention and gated-MLP updates through separate skip paths. RoPE and GQA live inside attention; the next checkpoints inspect their interfaces.

![Explain both residual updates](../figures/chapter-05/day-07-02_architecture_defense-architecture-detail.png)

*Saved reference preview. The next cell regenerates the figure; it does not overwrite this preview.*

In [ ]:
show_visual(architecture.block_detail(focus='modern', modern=True))

## 1. Trace the actual computation

Which dimensions change, and which must stay fixed so residual addition remains legal? Why is an attention output projection different from the vocabulary head even if both happen to end in width 16 here?

**Your prediction:** _Explain the shape path before running the trace._

In [ ]:
# Optional attempt: write the expected shapes from token lookup to logits.

### Reference solution

Temporary forward hooks record actual module boundaries. Linear Q/K/V outputs here are **before** the split into heads. DecoderBlock rows record the state, not the returned cache.

In [ ]:
trace = trace_decoder(model, ids)
for row in trace:
    print(row["module"], row["input_shape"], "->", row["output_shape"])
assert all(row["finite"] for row in trace)
by_name = {row["module"]: row for row in trace}
assert by_name["token"]["output_shape"] == [2, 6, 16]
assert by_name["blocks.0.attn.q"]["output_shape"] == [2, 6, 16]
assert by_name["blocks.0.attn.k"]["output_shape"] == [2, 6, 8]
assert by_name["blocks.0.mlp.up"]["output_shape"] == [2, 6, 32]
assert by_name["lm_head"]["output_shape"] == [2, 6, 16]
assert model.lm_head.weight is model.token.weight
assert all(not module._forward_hooks for module in model.modules())

### Why this works

Four query heads of width four produce 16 projected query coordinates, but two KV heads produce only eight coordinates before reshaping. MLP width expands to 32 and returns to D=16. The final size 16 means vocabulary candidates, not hidden features: change vocab without changing width to separate those roles. Hooks are removed after tracing, including on errors.

## 2. Reconcile the budget with stored tensors

If we halve KV heads, which part of the model shrinks? Does the entire parameter count halve? Why should the tied head be counted only once?

**Your prediction:** _Name the affected and unaffected components._

In [ ]:
# Optional attempt: count unique model parameters and compact cache payload.

### Reference solution

In [ ]:
ledger = parameter_ledger(model)
estimate = cost_estimate(cfg, batch=2, length=6, bytes_per_element=8)
with torch.no_grad():
    _, full_cache = model(ids, return_cache=True)
payload = sum(t.numel()*t.element_size() for pair in full_cache for t in pair)
assert sum(ledger.values()) == parameter_count(model) == estimate["parameters"] == 4960
assert payload == estimate["logical_kv_bytes"] == 3072
assert estimate["dense_forward_matmul_flops"] == 125952
print(ledger)
print(estimate)
print("Measured compact tensor payload:", payload, "bytes")

### Why this works

Unique Parameter objects determine storage count. Tied lookup/output weights count once. Cache payload is a sum of tensor element counts times their element sizes; it is not peak process memory. The FLOP estimate counts dense forward matrix products (multiply-add=2), not elapsed time or backward work. These checks cannot select the best trained architecture.

### Where the parameter budget goes

Bars use the actual unique-parameter ledger above. Attention includes the optional Q/K norm scales; the tied vocabulary matrix belongs to the token group. Rerun after changing a configuration and rebuilding the model.

![Where the parameter budget goes](../figures/chapter-05/day-07-02_architecture_defense-visual-budget.png)

*Saved reference preview. The next cell regenerates the figure; it does not overwrite this preview.*

In [ ]:
show_visual(viz.parameter_budget(ledger))

## 3. Follow one loss into the trainable components

Does backward update the parameters? Does a nonzero gradient tell us that a component has learned a useful linguistic role?

**Your prediction:** _Distinguish connectivity, sensitivity, an optimizer step, and capability._

In [ ]:
# Optional attempt: snapshot parameters, compute next-token loss, and inspect gradients.

### Reference solution

In [ ]:
before = {name: p.detach().clone() for name, p in model.named_parameters()}
model.zero_grad(set_to_none=True)
loss = next_token_loss(model(ids), labels)
loss.backward()
gradient_rows = [
    {"name": name, "connected": p.grad is not None,
     "finite": p.grad is not None and bool(torch.isfinite(p.grad).all()),
     "norm": None if p.grad is None else float(p.grad.norm())}
    for name, p in model.named_parameters()
]
assert all(row["connected"] and row["finite"] for row in gradient_rows)
assert all(torch.equal(p.detach(), before[name]) for name, p in model.named_parameters())
print("Mean next-token loss:", float(loss.detach()))
print("Connected finite parameter tensors:", len(gradient_rows))
for row in gradient_rows[:8]:
    print(row)
print("No optimizer.step(): stored parameter values are unchanged.")

### Why this works

Twelve aligned target losses are averaged and backpropagated through shared parameters. A connected finite gradient verifies a path for this fixture; it need not be nonzero in every situation, and its norm is not a comparable importance score across differently scaled parameters. True weight tying sums lookup and classifier contributions. Backward changes gradient buffers, not weight values. The fixture is untrained and simple enough for successor memorization; it does not establish contextual understanding.

## 4. Diagnose three mystery cases

You receive three systems: A, B, and C. All may produce finite logits. Available evidence is future-token invariance and cached/full-prefix agreement. What pattern would suggest a wrong position offset? What pattern would suggest a noncausal operation outside attention?

**Your prediction:** _Describe the expected pass/fail patterns before opening the reference. Then inspect the implementation to distinguish evidence from a unique diagnosis._

In [ ]:
# Optional attempt: write your predicted evidence signatures for each kind of fault.

### Reference solution — reveal the controlled faults

A is unchanged. B restarts the suffix's RoPE offset at zero. C subtracts a full-time mean in the first normalization position; attention still has its causal mask. These are deliberately constructed faults, not a claim that this test signature uniquely identifies every real bug.

In [ ]:
leaky = copy.deepcopy(model)
leaky.blocks[0].norm1 = WrongAxisCenter()
reports = {
    "A: reference": boundary_report(model, ids),
    "B: wrong offset": boundary_report(model, ids, rope_offset=0),
    "C: time mixing": boundary_report(leaky, ids),
}
assert all(reports["A: reference"][key] for key in ("finite", "causal", "cache"))
assert reports["B: wrong offset"]["causal"] and not reports["B: wrong offset"]["cache"]
assert reports["C: time mixing"]["finite"] and not reports["C: time mixing"]["causal"]
assert not reports["C: time mixing"]["cache"]
for name, result in reports.items():
    print(name, result)

### Why this works

B preserves causal visibility but compares newly rotated Q/K against cached keys with inconsistent position indices. C makes an early state depend on later inputs via its time mean, and prefix versus full-sequence statistics differ. A finite loss alone detects neither defect. These controlled examples support hypotheses, not exhaustive diagnosis: inspect the operation and reproduce the smallest failing input before deciding a fix.

### Separate numerical health from causal and cache correctness

Each tile reports the actual assertions for this tiny fixture. PASS does not mean overall model quality; FAIL identifies a violated tested property. Color is redundant with the printed labels.

![Separate numerical health from causal and cache correctness](../figures/chapter-05/day-07-02_architecture_defense-visual-diagnostics.png)

*Saved reference preview. The next cell regenerates the figure; it does not overwrite this preview.*

In [ ]:
show_visual(viz.audit_outcomes(reports))

## 5. Write a defensible comparison contract

Suppose we apply one shared block twice instead of once. What stays fixed, what increases, and what result would justify the extra computation? Which measurements must be taken rather than estimated?

**Your proposal:** _Specify the hypothesis, matching condition, evaluation, controls, and failure rules. This is a design exercise; do not start a long training run._

In [ ]:
# Optional attempt: proposed_contract = {...}; keep parameter-, compute-, and wall-clock-matching distinct.

### Reference solution — one possible contract, not a universal recipe

In [ ]:
proposed_contract = {
    "status": "proposal only; not executed or approved for a long run",
    "hypothesis": "At fixed stored parameters, a trained two-use core may reduce held-out next-token loss.",
    "matching": "stored parameter count; NOT equal FLOPs or wall-clock time",
    "variants": ["one shared core application", "two shared core applications"],
    "controls": ["same tokenizer and frozen data splits", "same token budget and declared seed set",
                 "same precision, optimizer recipe, and evaluation sampling settings"],
    "measure": ["held-out loss", "effective block applications", "measured decode latency",
                "peak memory with a declared method", "fixed prompt samples", "actual training compute/time"],
    "failure_rules": ["report nonfinite losses or gradients", "reject causal/cache errors",
                      "retain negative quality results", "do not pick seeds on the test set"],
    "not_proven": ["free compute", "universal recurrence superiority", "adaptive token routing",
                   "first-pass KV is reusable merely because weights are shared"],
}
assert "NOT equal FLOPs" in proposed_contract["matching"]
for key, value in proposed_contract.items():
    print(key, ":", value)

### Why this works

Parameter sharing fixes stored size while extra applications spend computation; matching parameters does not match FLOPs or latency. Define numeric budgets, dataset revisions, seed values, and measurement procedures before promoting this proposal to a runnable experiment. The [optional recurrence notebook](01_recurrent_depth.ipynb) tests weight identity and gradient accumulation, not this quality hypothesis.

## Architecture defense — explain back, then decide what remains

Use the trace and diagnostic evidence to explain one token's route from lookup to logits. Identify where positions enter, why the stream width stays fixed, where gradients accumulate, and which evidence would change your design decision.

**Your explanation:** _Record it here._

**Remaining uncertainty / next controlled test:** _Record it here._

Reference rubric: a defensible answer includes **choice → mechanism → tensor shapes → evidence → trade-off → failure risk → next experiment**. Merely running all cells does not complete the defense. Use [Chapter 5](../../book/chapters/05-building-a-modern-decoder.md) and the [solution guide](../../book/solutions/05-decoder-notebook-solutions.md).

This notebook is a bounded CPU architecture audit. It does not train a new model, allocate a large candidate, measure GPU serving performance, or render animations. All animation production remains on Mac Studio after approval.